# GGIS 407 - Lab 5: PostgreSQL
#### Interactive Jupyter Notebook

In this notebook we will use <b>PostgreSQL</b> to learn the basic concepts of databases. We will also use <b>Structured Query Language (SQL)</b> to query and manipulate data in the database.

## Notebook Outline:
- [Introduction](#intro)
- [Install PostgreSQL](#run_once) (first time only)
- [Setup](#setup) (run this first!)
- [Database Basics](#postgreSQL)
    - [Create Database](#create_db)
    - [Connect to Database](#connect_db)
    - [View Databases and Tables](#view_db)
- [Database Operations](#op_db)
    - [Create a Table](#create_table)
    - [Insert Data into Table](#insert_data)
    - [List Contents of the Table](#list_data)
    - [Update Data in Table](#update_table)
    - [Delete Data in Table](#delete_d)
    - [Sort Table](#sort_table)
    - [Aggregate Data](#count_row)
    - [Export Data to a CSV file](#export_data)
    - [Import Data from a CSV file](#import_data)
- [Join Operations](#join)
    - [Natural Join](#nt_join)
    - [Inner Join](#in_join)
    - [Outer Joins](#out_join)

<a id='intro'></a>
## Introduction

<b>PostgreSQL</b> is a powerful, widely used, open source relational database system. It uses and extends the SQL language, with many features that safely store and scale complicated data workloads. PostgreSQL is designed for reliability and data integrity, has a robust feature set, is extensible (PostGIS, which you'll use in the next notebook, is an extension), and is ACID compliant.

Useful documentation:

>PostgreSQL: https://www.postgresql.org/
<br>PostgreSQL Documentation: https://www.postgresql.org/docs/current/
<br>PostgreSQL Tutorial: https://www.postgresql.org/docs/current/tutorial.html
<br>jupysql (the `%sql` commands): https://jupysql.ploomber.io/

<a id='run_once'></a>
## Install PostgreSQL (first time only)
This lab runs its own small PostgreSQL database server inside your I-GUIDE JupyterHub account. The next cell installs what it needs:

- the PostgreSQL server and the PostGIS extension, into a folder called `pgenv` in your home directory (**this takes a few minutes the first time**; after that it is skipped), and
- two Python packages: `jupysql`, which provides the `%sql` commands used in this notebook, and `psycopg2`, which lets Python talk to PostgreSQL.

If the cell says it installed or removed Python packages, choose <b>Kernel -> Restart Kernel</b>, then continue with Setup.

In [ ]:
import os, shutil, subprocess
from importlib.metadata import version, PackageNotFoundError

def installed(pkg):
    try:
        version(pkg)
        return True
    except PackageNotFoundError:
        return False

changed = False

# 1. Python packages for the %sql commands
if installed('ipython-sql'):   # an older %sql package that conflicts with jupysql
    get_ipython().run_line_magic('pip', 'uninstall -y -q ipython-sql')
    changed = True
needed = []
if not installed('jupysql'):
    needed.append('jupysql')
if not (installed('psycopg2') or installed('psycopg2-binary')):
    needed.append('psycopg2-binary')
if needed:
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(needed))
    changed = True

# 2. PostgreSQL server and PostGIS, installed once into ~/pgenv
PGENV = os.path.expanduser('~/pgenv')
if not os.path.exists(os.path.join(PGENV, 'bin', 'pg_ctl')):
    installer = shutil.which('mamba') or shutil.which('conda')
    print('Installing PostgreSQL and PostGIS into ~/pgenv. This takes a few minutes the first time...')
    subprocess.run([installer, 'create', '-y', '-q', '-p', PGENV, '-c', 'conda-forge',
                    'postgresql=16', 'postgis'], check=True)

if changed:
    print('Python packages were installed or removed: choose Kernel -> Restart Kernel, then run Setup.')
else:
    print('Everything is installed. Continue with Setup.')

<a id='setup'></a>
## Setup
Run this cell every time you open the notebook. It creates the database cluster the first time (in a folder called `pgdata` in your home directory) and starts the PostgreSQL server if it isn't already running. Both Lab 5 notebooks use the same server.

In [ ]:
# set up and start the PostgreSQL server
import os, subprocess

PGENV = os.path.expanduser('~/pgenv')
os.environ['PATH'] = os.path.join(PGENV, 'bin') + os.pathsep + os.environ['PATH']
os.environ['PGDATA'] = os.path.expanduser('~/pgdata')   # where PostgreSQL keeps its files
os.environ['PGHOST'] = 'localhost'                      # the server runs on this machine
os.environ['PGPORT'] = '5432'                           # PostgreSQL's standard port
pgdata = os.environ['PGDATA']

# create the database cluster (only the first time)
if not os.path.exists(os.path.join(pgdata, 'PG_VERSION')):
    !initdb --auth=trust --encoding=UTF8 --no-locale > /dev/null && echo "Database cluster created in ~/pgdata"

# start the server if it isn't running yet
if subprocess.run(['pg_isready', '-q']).returncode == 0:
    print('PostgreSQL server is already running.')
else:
    # a lock file left over from an earlier session can stop the server from starting
    pid_file = os.path.join(pgdata, 'postmaster.pid')
    if os.path.exists(pid_file):
        os.remove(pid_file)
    !pg_ctl --log={pgdata}/server.log --wait start

# load the %sql commands and show up to 20 rows of each result
%reload_ext sql
%config SqlMagic.displaylimit = 20

<a id='postgreSQL'></a>
## Database Basics

<a id='create_db'></a>
### Create Database

A single PostgreSQL server can hold many databases. `createdb` is a command-line tool that comes with PostgreSQL; the `!` at the start of the line runs it as a shell command.

In [ ]:
# Create a new database called postgresql_lab5
# (if you run this again, it reports that the database already exists; that's fine)
!createdb postgresql_lab5

<a id='connect_db'></a>
### Connect to Database

The `%sql` command connects to a database using a <b>connection string</b>: the kind of database and the Python driver to use (`postgresql+psycopg2`), the computer it runs on (`localhost` means this one), the port, and the database name.

In [ ]:
# Establish a connection to the database you have created
%sql postgresql+psycopg2://localhost:5432/postgresql_lab5

<a id='view_db'></a>
### View Databases and Tables

`psql` is PostgreSQL's command-line client. Its <b>meta-commands</b> start with a backslash: `\l` lists the databases on the server and `\dt` lists the tables in a database.

In [ ]:
# View all the databases on the server
!psql -d postgresql_lab5 -c "\l"

In [ ]:
# List all the tables in the database (there are none yet)
!psql -d postgresql_lab5 -c "\dt"

<a id='op_db'></a>
## Database Operations

<a id='create_table'></a>
### Create a Table

Create a new table called myTable in the postgresql_lab5 database.

In [ ]:
%%sql
-- Starting a cell with %%sql makes the whole cell SQL code; -- starts a comment in SQL
-- Create a new table called myTable, deleting any old table with the same name first
drop table if exists myTable;
create table myTable(key int, attr varchar(20), value float);

Notice that PostgreSQL turns unquoted names into lower case, so the table is stored as `mytable`.

In [ ]:
# Check that the table was created
!psql -d postgresql_lab5 -c "\dt"

<a id='insert_data'></a>
### Insert Data into Table

Load a number of data rows into the table.

In [ ]:
# Insert a row of data into the table
%sql insert into myTable values(1, 'attr0', 100);

In [ ]:
# View the data in the table
%sql select * from myTable

In [ ]:
# Insert more rows. Note how we can list the columns in a different order
%sql insert into myTable values(2, 'attr1', 101);
%sql insert into myTable(key, attr, value) values(3, 'attr1', 102);
%sql insert into myTable(attr, key, value) values('attr1', 4, 103);
%sql insert into myTable(value, key, attr) values(106, 5, 'attr1');

<a id='list_data'></a>
### List Contents of the Table

In [ ]:
# Review the contents of the table
%sql select * from myTable;

In [ ]:
# View the rows that meet the condition attr='attr1'
%sql select * from myTable where attr='attr1';

In [ ]:
# View the rows that meet the condition key=2
%sql select * from myTable where key=2;

In [ ]:
# View the key and value columns of one of the rows in the table
%sql select key, value from myTable limit 1;

Notice that the `select` statement lets you view the contents of the table, and the `where` clause lets you filter which records you want to view.

<a id='update_table'></a>
### Update Data in Table

In [ ]:
# Change the value of a data item
%sql update myTable set value=1 where key=1;

In [ ]:
# Change the value of another data item
%sql update myTable set value=105 where key=3;

<a id='delete_d'></a>
### Delete Data in Table

First we add a row so we have something to delete.

In [ ]:
# Add a row
%sql insert into myTable(value, key, attr) values(100, 1011, 'attr1');

In [ ]:
%sql select * from myTable

In [ ]:
# Delete the row with key 1011
%sql delete from myTable where key=1011;

In [ ]:
%sql select * from myTable

<a id="sort_table"></a>
### Sort Table

In [ ]:
# Sort the table in ascending order
%sql select * from myTable order by key asc;

In [ ]:
# Sort the table in descending order
%sql select * from myTable order by key desc;

<a id="count_row"></a>
### Aggregate Data

In [ ]:
# Find the number of rows in myTable
%sql select count(*) from myTable;

In [ ]:
# Find the number of rows in myTable where attr ends in the character 1
%sql select count(*) from myTable where attr like '%1';

In [ ]:
# Find the maximum of the column "value" in myTable
%sql select max(value) from myTable;

In [ ]:
# Find the minimum of the column "value" in myTable
%sql select min(value) from myTable;

In [ ]:
# Find the average of "value" where attr contains the character 1
# (ilike is a case-insensitive version of like)
%sql select avg(value) from myTable where attr ilike '%1%';

<a id="export_data"></a>
### Export Data to a CSV file

The psql `\copy` meta-command copies data between a table (or the result of a query) and a file on this computer.

In [ ]:
# Export the result of a query to myTable.csv, with a header row
!psql -d postgresql_lab5 -c "\copy (SELECT * FROM myTable) TO 'myTable.csv' WITH (FORMAT csv, HEADER)"

In [ ]:
# Look at the file we just created
print(open('myTable.csv').read())

<a id="import_data"></a>
### Import Data from a CSV file

First we create a small CSV file called myTable2.csv in the same folder as this notebook.

In [ ]:
# Write a small CSV file
with open('myTable2.csv', 'w') as f:
    f.write('key,attr1,value\n'
            '1,att1,1\n'
            '1,attr0,1\n'
            '2,attr0,20\n'
            '4,attr1,103\n'
            '6,att5,107\n')
print(open('myTable2.csv').read())

In [ ]:
# Each -c runs one command, in order:
# 1. drop any existing table named myTable2
# 2. create a new table with the columns we need
# 3. copy the data from the CSV file into the table (HEADER skips the first line)
# 4. count how many rows were imported
!psql -d postgresql_lab5 \
    -c "DROP TABLE IF EXISTS myTable2" \
    -c "CREATE TABLE myTable2(key int, attr1 varchar(20), value float)" \
    -c "\copy myTable2 FROM 'myTable2.csv' WITH (FORMAT csv, HEADER)" \
    -c "SELECT COUNT(*) FROM myTable2"

In [ ]:
# View the data in the new table
%sql select * from myTable2

<a id="join"></a>
## Join Operations

Let us first review the contents of the two tables.

In [ ]:
%sql select * from myTable

In [ ]:
%sql select * from myTable2

<a id="nt_join"></a>
### Natural Join

A <b>natural join</b> is a join operation that creates an implicit join clause for you based on the common columns in the two tables being joined: the columns that have the same name in both tables. Natural joins can be inner or outer joins, but by default they are inner joins.

In [ ]:
%sql select * from myTable natural join myTable2;

In [ ]:
%sql update myTable2 set value=104 where (key=1 and attr1 like '%0');

In [ ]:
%sql select * from myTable natural join myTable2;

<b>Explain the reason for the difference between the first and the second execution of the natural join.</b>

<a id="in_join"></a>
### Inner Join

An <b>inner join</b> selects all the records (rows) that have matching values in both tables in the columns that have been specified.

In [ ]:
%sql select myTable.*, myTable2.attr1, myTable2.value from myTable inner join myTable2 on myTable.key=myTable2.key;

In [ ]:
# An alternative way to write the same join
%sql select * from myTable, myTable2 where myTable.key=myTable2.key;

<b>Please provide a brief explanation of the result.</b>

<a id="out_join"></a>
### Outer Joins

An <b>outer join</b> also combines rows from two tables, but unlike an inner join, it returns every row from one or both tables, even when the join condition fails. There are three types of outer joins:
1. Left outer join
2. Right outer join
3. Full outer join

As their names suggest, a left outer join keeps all the rows of the left table even if there are no matches in the right table (rows of the right table that don't match are dropped), while a right outer join keeps all the rows of the right table even if there are no matches in the left table. Finally, a full outer join keeps all the rows of both tables. Where there is no match, the missing columns are filled with NULL (shown as None).

In [ ]:
# Left outer join
%sql select myTable.*, myTable2.* from myTable left outer join myTable2 on myTable.key=myTable2.key;

In [ ]:
# Right outer join
%sql select myTable.*, myTable2.* from myTable right outer join myTable2 on myTable.key=myTable2.key;

In [ ]:
# Full outer join
%sql select myTable.*, myTable2.* from myTable full outer join myTable2 on myTable.key=myTable2.key;

<b>Please provide a brief explanation of the result.</b>

## Try it
1. Add a column to myTable with `alter table myTable add column note varchar(50);` and fill it in for one row with `update`.
2. Write a query that returns the average `value` for each `attr` in myTable. (Hint: `group by`.)
3. Using an outer join, list the keys that appear in myTable2 but not in myTable.

In [ ]:
# Practice what you have learned about SQL
# Feel free to modify any code above